# LatinLibrary Reader Demo

## Setup

In [1]:
from latincyreaders import LatinLibraryReader, AnnotationLevel
from pprint import pprint
from itertools import islice

In [2]:
# Auto-downloads corpus on first use if not found
reader = LatinLibraryReader()

## File Discovery

In [3]:
reader.fileids()[:8]

['12tables.txt',
 '1644.txt',
 'abbofloracensis.txt',
 'abelard/dialogus.txt',
 'abelard/epistola.txt',
 'abelard/historia.txt',
 'addison/barometri.txt',
 'addison/burnett.txt']

In [8]:
len(reader.fileids())

2141

## Metadata

In [10]:
# Title extracted from first line of each file
sample_file = reader.fileids(match="lucan")[0]
title_line = next(reader.texts(sample_file)).split("\n")[0]
print(f"{sample_file}: {title_line[:80]}")

lucan/lucan1.txt: Lucan Liber I


## Core Interface

### texts()

In [13]:
next(reader.texts("lucan/lucan1.txt"))[:500]

('asque acies, et rupto foedere regni\n'
 'certatum totis concussi uiribus orbis                  5\n'
 'in commune nefas, infestisque obuia signis\n'
 'signa, pares aquilas et pila minantia pilis.\n'
 '      quis furor, o ciues, quae tanta licentia ferri?\n'
 'gentibus inuisis Latium praebere cruorem\n'
 'cumque superba foret Babylon spolianda tropaeis                  10\n'
 'Ausoniis umbraque erraret Crassus inulta\n'
 'bella geri plac')


### docs()

In [12]:
doc = next(reader.docs("lucan/lucan1.txt"))
doc

sque acies, et rupto foedere regni certatum totis concussi uiribus orbis 5 in commune nefas, infestisque obuia signis signa, pares aquilas et pila minantia pilis. quis furor, o ciues, quae tanta licentia ferri? gentibus inuisis Latium praebere cruorem cumque superba foret Babylon spolianda tropaeis 10 Ausoniis umbraque erraret Crassus inulta bella geri placuit nullos habitura triumphos? heu, quantum terrae potuit pelagique parari hoc quem ciuiles hauserunt sanguine dextrae, unde uenit Titan et n


In [35]:
print(f"tokens: {len(doc)}")
print(f"sents:  {len(list(doc.sents))}")
print(f"fileid: {doc._.fileid}")

tokens: 5730
sents:  236
fileid: lucan/lucan1.txt


### sents()

In [16]:
sent = next(reader.sents("lucan/lucan1.txt"))
sent

Lucan Liber I M. ANNAEI LVCANI BELLI CIVILIS LIBER PRIMVS Bella per Emathios plusquam ciuilia campos iusque datum sceleri canimus,

In [21]:
print(sent._.citation)

lucan/lucan1.txt:sent0


### tokens()

In [17]:
list(reader.tokens("lucan/lucan1.txt"))[:8]

[Lucan, Liber, I, M., ANNAEI, LVCANI, BELLI, CIVILIS]

In [18]:
tok = list(reader.tokens("lucan/lucan1.txt"))[10]
print(f"text: {tok.text}, lemma: {tok.lemma_}, pos: {tok.pos_}, tag: {tok.tag_}")

text: infestis, lemma: infestus, pos: ADJ, tag: adjective


## Latin Library Features

### paras()

In [15]:
# Skip early paragraphs to avoid paratextual material
paras = list(reader.paras("lucan/lucan1.txt"))
print(f"Total paragraphs: {len(paras)}")
for para in paras[4:7]:
    print(para.text[:100])
    print()

Total paragraphs: 4



### KWIC

In [26]:
# KWIC: "amor" with 5 tokens of context
catullus = reader.fileids(match="catullus")[0]
for hit in reader.kwic("amor", fileids=catullus, window=5, limit=5):
    print(f"{hit['left']} [{hit['match']}] {hit['right']}")
    print(f"  -- {hit['citation']}")
    print()

' hoc ut dixit , [Amor] sinistra ut ante dextra sternuit
  -- catullus.txt:4778

' hoc ut dixit , [Amor] sinistra ut ante dextra sternuit
  -- catullus.txt:4831

umquam contexit amores , nullus [amor] tali coniunxit foedere amantes ,
  -- catullus.txt:10675

sub Latmia saxa relegans dulcis [amor] gyro deuocet aereo : idem
  -- catullus.txt:11377

semper concordia uestras , semper [amor] sedes incolat assiduus . tu
  -- catullus.txt:11976



In [27]:
# KWIC by lemma — finds all forms (amo, amat, amant, amavit ...)
for hit in reader.kwic("amo", fileids=catullus, by_lemma=True, window=4, limit=5):
    print(f"{hit['left']} [{hit['match']}] {hit['right']}")
    print(f"  -- {hit['citation']}")
    print()

plus illa oculis suis [amabat] . nam mellitus erat
  -- catullus.txt:307

mea Lesbia , atque [amemus] , rumores que senum
  -- catullus.txt:565

uentitabas quo puella ducebat [amata] nobis quantum amabitur nulla
  -- catullus.txt:854

ducebat amata nobis quantum [amabitur] nulla . ibi illa
  -- catullus.txt:857

bella ? quem nunc [amabis] ? cuius esse diceris
  -- catullus.txt:950



### Concordance

In [24]:
# Build a concordance: word -> list of citations where it appears
# Note: Without Tesserae citations, uses fileid:sentN format

catullus = reader.fileids(match="catullus")[0]
conc = reader.concordance(fileids=catullus, basis="lemma")
print(f"Unique lemmas in {catullus}: {len(conc)}")

# Look up a specific lemma
if "amor" in conc:
    print("Citations for 'amor':")
    for cit in conc["amor"][:8]:
        print(f"  {cit}")

Unique lemmas in catullus.txt: 3446

Citations for 'amor':
  catullus.txt:746
  catullus.txt:801
  catullus.txt:1051
  catullus.txt:1399
  catullus.txt:1590
  catullus.txt:1812
  catullus.txt:2282
  catullus.txt:3297


In [25]:
# Concordance by surface form (exact spelling)
conc_text = reader.concordance(fileids=catullus, basis="text")

# Different forms of 'puella'
for form in ["puella", "puellae", "puellam", "puellas", "puellis"]:
    if form in conc_text:
        print(f"  {form}: {len(conc_text[form])} occurrences")

Occurrences of 'puella' forms:
  puella: 19 occurrences
  puellae: 18 occurrences
  puellam: 2 occurrences
  puellis: 2 occurrences


### FileSelector API

In [42]:
selection = reader.select().match(r"vergil")
print(f"Vergil files: {len(selection)}")
print(selection.preview(5))

Vergil files: 26
['vergil/aen1.txt', 'vergil/aen2.txt', 'vergil/aen3.txt', 'vergil/aen4.txt', 'vergil/aen5.txt']


In [43]:
selection = reader.select().match(r"cicero")
print(f"Cicero files: {len(selection)}")

for doc in islice(reader.docs(selection), 2):
    print(f"{doc._.fileid}: {len(list(doc.sents))} sentences")

Cicero files: 138
cicero/acad.txt: 236 sentences


cicero/adbrutum1.txt: 620 sentences


### Annotation Levels

In [40]:
# AnnotationLevel controls how much NLP processing to apply

# NONE - use texts() for raw strings (fastest)
# TOKENIZE - tokenization + sentence boundaries only
# BASIC - adds lemmatization and POS tagging (default)
# FULL - full pipeline including NER and dependency parsing

reader_fast = LatinLibraryReader(annotation_level=AnnotationLevel.TOKENIZE)
reader_full = LatinLibraryReader(annotation_level=AnnotationLevel.FULL)

print("Available annotation levels:")
for level in AnnotationLevel:
    print(f"  {level.name}: {level.value}")

Available annotation levels:
  NONE: none
  TOKENIZE: tokenize
  BASIC: basic
  FULL: full
